# Week 2 - Data Exploration

## Objective and business decision

This notebook explores whether the CRMLS sold-property dataset for January 2025 through May 2026 is sufficiently complete, consistent, and interpretable for Week 2 descriptive analysis of California SingleFamilyResidence close prices.

**Business decision supported:** help listing teams and pricing analysts understand whether the full available January 2025 through May 2026 sold-property history is reliable enough to support pricing benchmarks and market prioritization.

**Week 2 scope:** data validation, feasibility assessment, and exploratory analysis only: load the fixed-period data, apply required population filters, inspect distributions, and flag data quality risks.

**Out of scope for Week 2:** model development, split strategy, dataset contract design, imputation decisions, feature selection, and algorithm comparison.

## 1. Setup

The notebook uses only pandas/numpy plus lightweight inline SVG plots so it can run without matplotlib or seaborn.

In [1]:
from pathlib import Path
from html import escape
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

RAW_DATA_DIR = Path("data/raw data")
FILE_PATTERN = "CRMLSSold*.csv"
EXPECTED_MONTHS = pd.period_range("2025-01", "2026-05", freq="M").astype(str).tolist()

TARGET_FILTERS = {
    "PropertyType": "Residential",
    "PropertySubType": "SingleFamilyResidence",
}

REQUIRED_COLUMNS = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "PropertyType",
    "PropertySubType",
    "CloseDate",
    "ListingContractDate",
    "PostalCode",
    "CountyOrParish",
    "MlsStatus",
]

NUMERIC_COLUMNS = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
]

DATE_COLUMNS = ["CloseDate", "ListingContractDate"]


def show_df(df, max_rows=20):
    """Display a dataframe in notebooks or print it during script-style validation."""
    try:
        from IPython.display import display
        display(df.head(max_rows))
    except Exception:
        print(df.head(max_rows).to_string(index=False))


def show_html(html):
    """Render HTML in notebooks or print it during script-style validation."""
    try:
        from IPython.display import HTML, display
        display(HTML(html))
    except Exception:
        print(html)


def source_month_from_filename(path):
    match = re.search(r"(20\d{2})(\d{2})", path.name)
    if not match:
        return pd.NA
    return f"{match.group(1)}-{match.group(2)}"


def pct(n, d):
    return np.nan if d == 0 else 100 * n / d

## 2. Load monthly CRMLS files

Load the fixed Week 2 dataset range: January 2025 through May 2026. Each monthly sold file receives a `source_month` value from the filename so full-period coverage can be validated explicitly.

In [2]:
csv_files = sorted(RAW_DATA_DIR.glob(FILE_PATTERN))

expected_files = {f"CRMLSSold{month.replace('-', '')}.csv" for month in EXPECTED_MONTHS}
actual_files = {path.name for path in csv_files}
missing_files = sorted(expected_files - actual_files)
unexpected_files = sorted(actual_files - expected_files)

if missing_files or unexpected_files:
    raise ValueError(
        "Week 2 requires the fixed January 2025 through May 2026 file set. "
        f"Missing files: {missing_files}. Unexpected files: {unexpected_files}."
    )

loaded_frames = []
file_inventory = []

for file_path in csv_files:
    frame = pd.read_csv(file_path, low_memory=False)
    frame["source_file"] = file_path.name
    frame["source_month"] = source_month_from_filename(file_path)
    loaded_frames.append(frame)
    file_inventory.append(
        {
            "source_file": file_path.name,
            "source_month": source_month_from_filename(file_path),
            "rows": len(frame),
            "columns": frame.shape[1] - 2,
        }
    )

raw_df = pd.concat(loaded_frames, ignore_index=True, sort=False)
file_inventory_df = pd.DataFrame(file_inventory)

print(f"Loaded fixed Week 2 range: {EXPECTED_MONTHS[0]} through {EXPECTED_MONTHS[-1]} ({len(csv_files)} monthly files).")
print(f"Raw combined shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns")
show_df(file_inventory_df, max_rows=30)

Loaded fixed Week 2 range: 2025-01 through 2026-05 (17 monthly files).
Raw combined shape: 352,354 rows x 82 columns


,source_file,source_month,rows,columns
0,CRMLSSold202501.csv,2025-01,18738,80
1,CRMLSSold202502.csv,2025-02,18702,78
2,CRMLSSold202503.csv,2025-03,21445,78
3,CRMLSSold202504.csv,2025-04,23262,78
4,CRMLSSold202505.csv,2025-05,23154,78
5,CRMLSSold202506.csv,2025-06,22883,78
6,CRMLSSold202507.csv,2025-07,23646,78
7,CRMLSSold202508.csv,2025-08,22972,78
8,CRMLSSold202509.csv,2025-09,10827,78
9,CRMLSSold202510.csv,2025-10,23233,78


## 3. Schema and column validation

Validate that the Week 2 target, core features, filters, and context fields exist before any EDA. Missing required fields would block stakeholder-ready analysis.

In [3]:
schema_check = pd.DataFrame(
    {
        "column": REQUIRED_COLUMNS,
        "present": [col in raw_df.columns for col in REQUIRED_COLUMNS],
        "role": [
            "target" if col == "ClosePrice" else
            "core_feature" if col in NUMERIC_COLUMNS else
            "filter" if col in TARGET_FILTERS else
            "context"
            for col in REQUIRED_COLUMNS
        ],
    }
)

show_df(schema_check, max_rows=50)

missing_required = schema_check.loc[~schema_check["present"], "column"].tolist()
if missing_required:
    raise ValueError(f"Missing required Week 2 columns: {missing_required}")

,column,present,role
0,ClosePrice,True,target
1,LivingArea,True,core_feature
2,BedroomsTotal,True,core_feature
3,BathroomsTotalInteger,True,core_feature
4,LotSizeSquareFeet,True,core_feature
5,PropertyType,True,filter
6,PropertySubType,True,filter
7,CloseDate,True,context
8,ListingContractDate,True,context
9,PostalCode,True,context


In [4]:
working_df = raw_df.copy()

for col in NUMERIC_COLUMNS:
    working_df[col] = pd.to_numeric(working_df[col], errors="coerce")

for col in DATE_COLUMNS:
    working_df[col] = pd.to_datetime(working_df[col], errors="coerce")

print("Numeric/date parsing complete. Rows are not dropped in Week 2; invalid values are flagged for later preprocessing decisions.")

Numeric/date parsing complete. Rows are not dropped in Week 2; invalid values are flagged for later preprocessing decisions.


## 4. Required Residential + SingleFamilyResidence filter

The project target population is restricted to California residential single-family residences. This is a hard business and technical constraint, not an optional EDA slice.

In [5]:
filter_mask = np.ones(len(working_df), dtype=bool)
for col, value in TARGET_FILTERS.items():
    filter_mask &= working_df[col].astype("string").str.strip().eq(value)

sfh_df = working_df.loc[filter_mask].copy()

row_count_summary = pd.DataFrame(
    [
        {"stage": "raw_loaded", "rows": len(working_df), "share_of_raw_pct": 100.0},
        {
            "stage": "filtered_residential_single_family",
            "rows": len(sfh_df),
            "share_of_raw_pct": pct(len(sfh_df), len(working_df)),
        },
    ]
)

show_df(row_count_summary)

assert set(sfh_df["PropertyType"].dropna().unique()) == {"Residential"}, "Unexpected PropertyType after filtering."
assert set(sfh_df["PropertySubType"].dropna().unique()) == {"SingleFamilyResidence"}, "Unexpected PropertySubType after filtering."

print(f"Filtered dataset shape: {sfh_df.shape[0]:,} rows x {sfh_df.shape[1]:,} columns")

,stage,rows,share_of_raw_pct
0,raw_loaded,352354,100.000000
1,filtered_residential_single_family,175188,49.719316


Filtered dataset shape: 175,188 rows x 82 columns


In [6]:
filter_breakdown = (
    working_df.groupby(["PropertyType", "PropertySubType"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .head(15)
)

show_df(filter_breakdown, max_rows=15)

,PropertyType,PropertySubType,rows
51,Residential,SingleFamilyResidence,175188
83,ResidentialLease,SingleFamilyResidence,39431
40,Residential,Condominium,38058
72,ResidentialLease,Condominium,23295
55,Residential,Townhouse,13413
34,Land,NaN,10995
36,ManufacturedInPark,NaN,8774
70,ResidentialLease,Apartment,8198
87,ResidentialLease,Townhouse,6778
46,Residential,ManufacturedOnLand,3057


## 5. Missingness and invalid-value checks

These checks identify risk. They do not drop, impute, or correct rows in Week 2.

In [7]:
missingness = pd.DataFrame(
    {
        "column": REQUIRED_COLUMNS,
        "missing_rows": [sfh_df[col].isna().sum() for col in REQUIRED_COLUMNS],
        "missing_pct": [pct(sfh_df[col].isna().sum(), len(sfh_df)) for col in REQUIRED_COLUMNS],
        "non_missing_rows": [sfh_df[col].notna().sum() for col in REQUIRED_COLUMNS],
    }
).sort_values("missing_pct", ascending=False)

show_df(missingness, max_rows=50)

,column,missing_rows,missing_pct,non_missing_rows
4,LotSizeSquareFeet,3056,1.744412,172132
1,LivingArea,92,0.052515,175096
3,BathroomsTotalInteger,20,0.011416,175168
9,PostalCode,2,0.001142,175186
0,ClosePrice,0,0.000000,175188
2,BedroomsTotal,0,0.000000,175188
5,PropertyType,0,0.000000,175188
6,PropertySubType,0,0.000000,175188
7,CloseDate,0,0.000000,175188
8,ListingContractDate,0,0.000000,175188


In [8]:
invalid_checks = {
    "ClosePrice_missing": sfh_df["ClosePrice"].isna(),
    "ClosePrice_non_positive": sfh_df["ClosePrice"].le(0),
    "ClosePrice_implausibly_high_gt_50m": sfh_df["ClosePrice"].gt(50_000_000),
    "LivingArea_missing": sfh_df["LivingArea"].isna(),
    "LivingArea_non_positive": sfh_df["LivingArea"].le(0),
    "LivingArea_implausibly_high_gt_20000": sfh_df["LivingArea"].gt(20_000),
    "Bedrooms_missing": sfh_df["BedroomsTotal"].isna(),
    "Bedrooms_negative": sfh_df["BedroomsTotal"].lt(0),
    "Bedrooms_implausibly_high_gt_20": sfh_df["BedroomsTotal"].gt(20),
    "Bathrooms_missing": sfh_df["BathroomsTotalInteger"].isna(),
    "Bathrooms_negative": sfh_df["BathroomsTotalInteger"].lt(0),
    "Bathrooms_implausibly_high_gt_20": sfh_df["BathroomsTotalInteger"].gt(20),
    "LotSizeSquareFeet_missing": sfh_df["LotSizeSquareFeet"].isna(),
    "LotSizeSquareFeet_non_positive": sfh_df["LotSizeSquareFeet"].le(0),
    "LotSizeSquareFeet_implausibly_high_gt_1m": sfh_df["LotSizeSquareFeet"].gt(1_000_000),
}

invalid_summary = pd.DataFrame(
    [
        {
            "check": check_name,
            "flagged_rows": int(mask.fillna(False).sum()),
            "flagged_pct": pct(int(mask.fillna(False).sum()), len(sfh_df)),
        }
        for check_name, mask in invalid_checks.items()
    ]
).sort_values("flagged_rows", ascending=False)

show_df(invalid_summary, max_rows=50)

,check,flagged_rows,flagged_pct
12,LotSizeSquareFeet_missing,3056,1.744412
14,LotSizeSquareFeet_implausibly_high_gt_1m,463,0.264288
13,LotSizeSquareFeet_non_positive,94,0.053657
3,LivingArea_missing,92,0.052515
4,LivingArea_non_positive,78,0.044524
2,ClosePrice_implausibly_high_gt_50m,28,0.015983
9,Bathrooms_missing,20,0.011416
11,Bathrooms_implausibly_high_gt_20,10,0.005708
5,LivingArea_implausibly_high_gt_20000,7,0.003996
8,Bedrooms_implausibly_high_gt_20,3,0.001712


## 6. Distribution plots for target and core features

Plots are capped at the 99.5th percentile for readability unless noted. Capping is visualization-only; no rows are removed.

In [9]:
def summarize_numeric(df, col):
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    if series.empty:
        return pd.Series(dtype="float64")
    return series.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).round(2)


def svg_histogram(series, title, x_label, bins=40, cap_quantile=0.995, log_transform=False):
    clean = pd.to_numeric(series, errors="coerce").dropna()
    clean = clean[np.isfinite(clean)]

    if log_transform:
        clean = clean[clean > 0]
        clean = np.log10(clean)
        x_label = f"log10({x_label})"

    if clean.empty:
        return f"<p><b>{escape(title)}</b>: no valid numeric data to plot.</p>"

    upper = clean.quantile(cap_quantile)
    lower = clean.quantile(0.005)
    clipped = clean[(clean >= lower) & (clean <= upper)]
    counts, edges = np.histogram(clipped, bins=bins)

    width, height = 820, 320
    margin_left, margin_right, margin_top, margin_bottom = 70, 25, 45, 55
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    max_count = max(counts.max(), 1)
    bar_width = plot_width / len(counts)

    bars = []
    for i, count in enumerate(counts):
        bar_h = plot_height * count / max_count
        x = margin_left + i * bar_width
        y = margin_top + plot_height - bar_h
        bars.append(
            f'<rect x="{x:.2f}" y="{y:.2f}" width="{max(bar_width - 1, 1):.2f}" height="{bar_h:.2f}" fill="#2f6fb0" />'
        )

    tick_values = np.linspace(edges[0], edges[-1], 5)
    ticks = []
    for value in tick_values:
        x = margin_left + (value - edges[0]) / max(edges[-1] - edges[0], 1e-9) * plot_width
        label = f"{value:,.1f}" if log_transform else f"{value:,.0f}"
        ticks.append(f'<line x1="{x:.2f}" y1="{margin_top + plot_height}" x2="{x:.2f}" y2="{margin_top + plot_height + 6}" stroke="#333" />')
        ticks.append(f'<text x="{x:.2f}" y="{margin_top + plot_height + 24}" text-anchor="middle" font-size="11" fill="#333">{escape(label)}</text>')

    y_ticks = []
    for value in np.linspace(0, max_count, 4):
        y = margin_top + plot_height - (value / max_count) * plot_height
        y_ticks.append(f'<line x1="{margin_left - 6}" y1="{y:.2f}" x2="{margin_left}" y2="{y:.2f}" stroke="#333" />')
        y_ticks.append(f'<text x="{margin_left - 10}" y="{y + 4:.2f}" text-anchor="end" font-size="11" fill="#333">{int(value):,}</text>')

    return f'''
    <div style="margin: 12px 0 24px 0;">
      <svg width="{width}" height="{height}" role="img" aria-label="{escape(title)}">
        <rect x="0" y="0" width="{width}" height="{height}" fill="white" />
        <text x="{margin_left}" y="26" font-size="18" font-weight="700" fill="#111">{escape(title)}</text>
        <line x1="{margin_left}" y1="{margin_top + plot_height}" x2="{margin_left + plot_width}" y2="{margin_top + plot_height}" stroke="#333" />
        <line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{margin_top + plot_height}" stroke="#333" />
        {''.join(bars)}
        {''.join(ticks)}
        {''.join(y_ticks)}
        <text x="{margin_left + plot_width / 2}" y="{height - 10}" text-anchor="middle" font-size="12" fill="#333">{escape(x_label)}</text>
        <text x="18" y="{margin_top + plot_height / 2}" transform="rotate(-90 18 {margin_top + plot_height / 2})" text-anchor="middle" font-size="12" fill="#333">Listing count</text>
      </svg>
      <div style="font-size: 12px; color: #555;">Visualization range: 0.5th to {cap_quantile * 100:.1f}th percentile. Rows outside range are not removed.</div>
    </div>
    '''


def interpretation_block(title, what_it_says, decision_use, data_quality_risk):
    return f'''
    <div style="border-left: 4px solid #2f6fb0; padding: 8px 12px; margin: 0 0 22px 0; background: #f7f9fc;">
      <b>{escape(title)} - business interpretation</b>
      <ul>
        <li><b>What the distribution says:</b> {escape(what_it_says)}</li>
        <li><b>Decision relevance:</b> {escape(decision_use)}</li>
        <li><b>Data quality risk:</b> {escape(data_quality_risk)}</li>
      </ul>
    </div>
    '''


def plot_and_interpret(df, col, title, x_label, what_it_says, decision_use, data_quality_risk, log_transform=False):
    show_html(svg_histogram(df[col], title, x_label, log_transform=log_transform))
    summary = summarize_numeric(df, col)
    print(f"Summary for {col}:")
    print(summary.to_string())
    show_html(interpretation_block(title, what_it_says, decision_use, data_quality_risk))

In [10]:
plot_and_interpret(
    sfh_df,
    "ClosePrice",
    "ClosePrice distribution",
    "ClosePrice",
    "The full-period filtered dataset is strongly right-skewed: median ClosePrice is about $895K, the 75th percentile is about $1.43M, the 95th percentile is about $3.2M, and the visualization still extends to $8.8M at the 99.5th percentile. One non-positive ClosePrice and a maximum near $989.5M are data quality flags, not normal market behavior.",
    "Most observed sales are concentrated below roughly $1.5M, so pricing/listing analysis will be most statistically supported in that range. Luxury and ultra-luxury properties exist in the full-period data, but they are sparse relative to the core market.",
    "ClosePrice needs explicit outlier review in Week 3. The extreme maximum and one zero sale price should not be ignored, because they can distort averages and any later preprocessing decisions.",
)

Summary for ClosePrice:
count    1.751880e+05
mean     1.334493e+06
std      7.414932e+06
min      0.000000e+00
1%       2.350000e+05
5%       3.690000e+05
25%      6.250000e+05
50%      8.950000e+05
75%      1.429000e+06
95%      3.200000e+06
99%      6.400006e+06
max      9.895000e+08


In [11]:
plot_and_interpret(
    sfh_df,
    "ClosePrice",
    "Log10 ClosePrice distribution",
    "ClosePrice",
    "After log transformation, the main mass of ClosePrice becomes easier to read and appears centered around the typical six-figure to low-seven-figure California single-family market. The long tail is still visible, but it no longer dominates the chart.",
    "The log view confirms that the full-period data contains both mainstream and high-end price tiers. Any stakeholder summary should avoid using only the raw mean, because the mean is pulled upward by extreme high-price records.",
    "For Week 2, this is an EDA warning: report medians and percentiles alongside means. Do not make removal or transformation decisions here; carry the skew issue into Week 3 preprocessing.",
    log_transform=True,
)

Summary for ClosePrice:
count    1.751880e+05
mean     1.334493e+06
std      7.414932e+06
min      0.000000e+00
1%       2.350000e+05
5%       3.690000e+05
25%      6.250000e+05
50%      8.950000e+05
75%      1.429000e+06
95%      3.200000e+06
99%      6.400006e+06
max      9.895000e+08


In [12]:
plot_and_interpret(
    sfh_df,
    "LivingArea",
    "LivingArea distribution",
    "LivingArea",
    "LivingArea is concentrated in a plausible single-family range: median is about 1,814 sqft, the middle 50% is roughly 1,384 to 2,435 sqft, and the 95th percentile is about 3,822 sqft. However, 92 rows are missing LivingArea, 78 rows are non-positive, and the maximum is 56,500 sqft.",
    "This variable has strong coverage for full-period EDA, so it can support basic size-based interpretation of sale prices. The central range looks usable for describing typical California single-family homes.",
    "The zero/non-positive values and very large homes need data quality review. Week 2 should flag them only; Week 3 must decide whether they are valid estates, unit errors, or records requiring correction/removal.",
)

Summary for LivingArea:
count    175096.00
mean       2044.43
std        1042.59
min           0.00
1%          736.00
5%          966.00
25%        1384.00
50%        1813.50
75%        2435.00
95%        3822.00
99%        5656.05
max       56500.00


In [13]:
plot_and_interpret(
    sfh_df,
    "BedroomsTotal",
    "BedroomsTotal distribution",
    "BedroomsTotal",
    "BedroomsTotal is concentrated around standard single-family layouts: median is 3 bedrooms, the 25th to 75th percentile range is 3 to 4 bedrooms, and the 95th percentile is 5 bedrooms. There are 108 non-positive bedroom counts and an extreme maximum of 45 bedrooms.",
    "The full-period data is strongest for interpreting typical 3- to 4-bedroom homes. Those are likely the most stable segment for descriptive price comparisons by size and location.",
    "Zero-bedroom and 45-bedroom records are not credible without review. They should be listed as data quality exceptions rather than treated as normal residential inventory in Week 2 findings.",
)

Summary for BedroomsTotal:
count    175188.00
mean          3.49
std           0.97
min           0.00
1%            2.00
5%            2.00
25%           3.00
50%           3.00
75%           4.00
95%           5.00
99%           6.00
max          45.00


In [14]:
plot_and_interpret(
    sfh_df,
    "BathroomsTotalInteger",
    "BathroomsTotalInteger distribution",
    "BathroomsTotalInteger",
    "BathroomsTotalInteger is concentrated around 2 to 3 bathrooms: median is 2, the 75th percentile is 3, and the 95th percentile is 5. The field has low missingness with 20 missing rows, but 80 non-positive values and a maximum of 45 bathrooms are quality flags.",
    "For EDA, bathroom count is usable for describing the typical housing stock in the full-period filtered dataset. Most records fall into ordinary residential bathroom counts.",
    "The integer field may hide half-bath detail, and the non-positive/extreme values must be reviewed before any Week 3 cleaning decision. Week 2 should document the issue, not fix it.",
)

Summary for BathroomsTotalInteger:
count    175168.00
mean          2.63
std           1.14
min           0.00
1%            1.00
5%            1.00
25%           2.00
50%           2.00
75%           3.00
95%           5.00
99%           6.00
max          45.00


In [15]:
plot_and_interpret(
    sfh_df,
    "LotSizeSquareFeet",
    "LotSizeSquareFeet distribution",
    "LotSizeSquareFeet",
    "LotSizeSquareFeet is highly right-skewed: median is about 7,277 sqft, the 75th percentile is about 10,454 sqft, the 95th percentile is about 50,094 sqft, and the maximum exceeds 2.0B sqft. Missingness is the largest among the plotted fields at 3,056 rows, with 94 non-positive values.",
    "The central lot-size range is useful for describing typical single-family parcels, but the tail is too extreme to summarize with the mean alone. ZIP/county context matters because large-lot properties are not comparable across all California markets.",
    "Lot size has the highest immediate data quality risk among the core EDA variables. Week 2 should flag missing, zero, and extreme parcel sizes for Week 3 review rather than applying blanket exclusions.",
)

Summary for LotSizeSquareFeet:
count    1.721320e+05
mean     3.322812e+05
std      1.664879e+07
min      0.000000e+00
1%       1.750000e+03
5%       3.221000e+03
25%      5.663000e+03
50%      7.276580e+03
75%      1.045400e+04
95%      5.009400e+04
99%      2.640037e+05
max      2.087221e+09


## 7. Geographic and monthly coverage checks

Coverage determines whether insights can be trusted for pricing and listing decisions across California markets.

In [16]:
monthly_counts = (
    sfh_df.groupby("source_month", dropna=False)
    .size()
    .reset_index(name="filtered_rows")
    .sort_values("source_month")
)

print("Filtered row count by source month:")
show_df(monthly_counts, max_rows=50)

observed_months = monthly_counts["source_month"].astype(str).tolist()
if observed_months != EXPECTED_MONTHS:
    raise ValueError(
        "Filtered data does not cover the required fixed period in order. "
        f"Expected {EXPECTED_MONTHS}, observed {observed_months}."
    )

Filtered row count by source month:


,source_month,filtered_rows
0,2025-01,8144
1,2025-02,8851
2,2025-03,10610
3,2025-04,11880
4,2025-05,11777
5,2025-06,11701
6,2025-07,12114
7,2025-08,11454
8,2025-09,5162
9,2025-10,12029


In [17]:
zip_volume = (
    sfh_df.assign(PostalCode=sfh_df["PostalCode"].astype("string").str.strip())
    .groupby("PostalCode", dropna=False)
    .agg(
        rows=("ListingKey", "size"),
        median_close_price=("ClosePrice", "median"),
        median_living_area=("LivingArea", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
    .head(20)
)

county_volume = (
    sfh_df.groupby("CountyOrParish", dropna=False)
    .agg(
        rows=("ListingKey", "size"),
        median_close_price=("ClosePrice", "median"),
        median_living_area=("LivingArea", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
    .head(20)
)

print("Top ZIP codes by filtered record volume:")
show_df(zip_volume, max_rows=20)

print("Top counties by filtered record volume:")
show_df(county_volume, max_rows=20)

Top ZIP codes by filtered record volume:


,PostalCode,rows,median_close_price,median_living_area
441,92253,1343,894990.0,2468.0
485,92345,1030,463000.0,1758.0
566,92584,985,630000.0,2256.0
424,92223,935,535000.0,2022.0
572,92592,908,795000.0,2371.0
575,92596,875,619704.0,2151.0
557,92562,843,710000.0,2300.0
422,92211,822,575000.0,1930.0
479,92336,796,725000.0,2070.0
558,92563,756,710000.0,2563.0


Top counties by filtered record volume:


,CountyOrParish,rows,median_close_price,median_living_area
19,Los Angeles,43403,1015000.0,1726.0
35,Riverside,26929,630000.0,2000.0
39,San Diego,19355,1060000.0,1861.0
38,San Bernardino,18778,550000.0,1684.0
30,Orange,16907,1400000.0,2004.0
0,Alameda,8344,1300000.0,1700.0
6,Contra Costa,8218,898444.0,1945.0
58,Ventura,5709,969000.0,1905.5
45,Santa Clara,5290,1910000.0,1785.0
42,San Luis Obispo,2936,940000.0,1758.0


In [18]:
date_coverage = []
for col in DATE_COLUMNS:
    valid = sfh_df[col].dropna()
    date_coverage.append(
        {
            "date_field": col,
            "non_missing_rows": len(valid),
            "missing_pct": pct(sfh_df[col].isna().sum(), len(sfh_df)),
            "min_date": valid.min() if len(valid) else pd.NaT,
            "max_date": valid.max() if len(valid) else pd.NaT,
        }
    )

show_df(pd.DataFrame(date_coverage), max_rows=10)

,date_field,non_missing_rows,missing_pct,min_date,max_date
0,CloseDate,175188,0.0,2025-01-01,2026-05-31
1,ListingContractDate,175188,0.0,1984-08-28,2026-05-30


## 8. Week 2 findings, assumptions, limitations, and Week 3 risks

### Assumptions

- `BedroomsTotal` is the Week 2 bedrooms field.
- `BathroomsTotalInteger` is the Week 2 bathrooms field.
- `LotSizeSquareFeet` is the primary lot-size field.
- `ClosePrice` is the target variable, but Week 2 only explores it.
- No rows are dropped or imputed in this notebook except for visualization-only exclusion of nulls/non-finite values inside plots.

### Limitations

- EDA plots show historical sold records, not active listing performance.
- Distribution plots do not prove predictive power.
- Outlier thresholds are flags for review, not final exclusion rules.
- Geographic volume tables identify coverage, not market attractiveness by themselves.

### Decision implication

If required fields have acceptable coverage and distributions are plausible, the dataset can proceed to Week 3 preprocessing. If target, size, location, or date fields show high missingness or implausible values, Week 3 must document whether rows are removed, imputed, or flagged and how each choice affects pricing decisions.

### Week 3 risks to carry forward

- Missing `ClosePrice`, `LivingArea`, or location fields can reduce analysis reliability.
- Skewed `ClosePrice` requires segment-aware evaluation later.
- Sparse ZIP codes may be unreliable for geography-level decisions.
- Date-field quality affects whether monthly coverage and lifecycle timing can be trusted.
- Week 3 must decide how to handle missing, invalid, and extreme values; Week 2 only flags them.